In [1]:
from numpy.ma import core
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression 
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline, FeatureUnion, Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, classification_report,make_scorer,roc_curve, precision_score, recall_score, roc_curve, auc
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

data_train = pd.read_csv("data/app_train_processed.csv")
data_test = pd.read_csv("data/app_valid_processed.csv")

In [2]:
X_train = data_train.drop(columns=['TARGET'])
y_train = data_train['TARGET']
X_test = data_test.drop(columns=['TARGET'])
y_test = data_test['TARGET']

# Identyfikacja kolumn kategorycznych (tekstowych)
kolumny_kategoryczne = X_train.select_dtypes(include=['object']).columns.tolist()
kolumny_numeryczne = X_train.select_dtypes(exclude=['object']).columns.tolist()

# Preprocessing: OneHotEncoder dla kolumn tekstowych, reszta bez zmian
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), kolumny_numeryczne),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), kolumny_kategoryczne)
    ]
)

zbiory_treningowy=[X_train]
zbiory_testowy=[X_test]
etykiety_treningowe=[y_train]
etykiety_testowe=[y_test]
opisy=["Czysta próbka"]

nazwy = ["Regresja logistyczna", "Lasy losowe", 
         "Regresja logistyczna (bez regularyzacji)"]

modele = [
    [("preprocessor", preprocessor), ("model", LogisticRegression(max_iter=1000, solver='lbfgs'))],
    [("preprocessor", preprocessor), ("model", RandomForestClassifier(n_jobs=-1))],
    [("preprocessor", preprocessor), ("model", LogisticRegression(max_iter=1000, penalty=None))],
]

parametry = [
    {"model__C": [0.1, 1], "model__class_weight": [{0:1,1:1}, {0:1,1:20}],
     "model__l1_ratio": [0]},
    {"model__n_estimators": [50, 100], "model__max_depth": [5, 10],
     "model__class_weight": [{0:1,1:1}, {0:1,1:20}], 
     "model__min_samples_split": [10], "model__min_samples_leaf": [10]},
    {"model__class_weight": [{0:1,1:1}, {0:1,1:20}]},
]


def potencjalny_zysk_score(y_true, y_pred,avg_amount=5000,lgd=0.7,margin=0.03):
    false_positive=np.sum(np.logical_and(y_pred == 1, y_true == 0))
    false_negative=np.sum(np.logical_and(y_pred == 0, y_true == 1))
    no_obs=sum(y_true==0) #ilość klientów zdrowych
    potential_profit=avg_amount*margin*no_obs
    #strata - zarówno ze względu na złych klientów, jak i ze względu na odrzucenie dobrych i tym samym nie zarobieniu marży
    loss=avg_amount*lgd*false_negative+avg_amount*margin*false_positive 
    result=(potential_profit-loss)/(potential_profit)
    return result

potent_profit_score = make_scorer(potencjalny_zysk_score, greater_is_better=True)

model_names = []
sample_names = []
accuracy_train = []
accuracy_test = []
auc_train = []
auc_test = []
f1_train = []
f1_test = []
pot_profit_train = []
pot_profit_test = []
false_positive = []
false_negative = []
true_positive = []
true_negative = []
recall = []
precision = []
best_estimators = []

from scipy.stats import ks_2samp

def find_cramer_cutoff(y_true, y_proba):
    """Znajduje próg = delta Cramera (max |F_good(s) - F_bad(s)|)"""
    scores_good = y_proba[y_true == 0]
    scores_bad = y_proba[y_true == 1]
    
    # KS statystyka + optymalny próg
    thresholds = np.sort(np.unique(y_proba))
    ks_values = []
    
    for t in thresholds:
        fpr = np.mean(scores_good >= t)  # % dobrych powyżej progu
        tpr = np.mean(scores_bad >= t)   # % złych powyżej progu
        ks_values.append(abs(tpr - fpr))
    
    best_idx = np.argmax(ks_values)
    
    return thresholds[best_idx], ks_values[best_idx]


for opis, zbior_treningowy, zbior_testowy, y_tr, y_te in zip(opisy, zbiory_treningowy, zbiory_testowy, etykiety_treningowe, etykiety_testowe):
    print("================================================================================================")
    print(opis)
    print("================================================================================================")
    for nazwa, model, parametr in zip(nazwy, modele, parametry):
        print("*****************************************************")
        print(f"Obecnie wykonuje się model: {nazwa}")
        print("*****************************************************")
        potok=Pipeline(model)
        optymalny_model=GridSearchCV(potok, parametr, cv=3, scoring=potent_profit_score, n_jobs=-1)
        optymalny_model.fit(zbior_treningowy, y_tr)
        best_estimators.append(optymalny_model.best_estimator_)


        # Prawdopodobieństwa i optymalny próg (delta Cramera na train)
        probs_train = optymalny_model.best_estimator_.predict_proba(zbior_treningowy)[:, 1]
        probs_test = optymalny_model.best_estimator_.predict_proba(zbior_testowy)[:, 1]
        cutoff, _ = find_cramer_cutoff(y_tr, probs_train)
        y_pred_train = (probs_train >= cutoff).astype(int)
        y_pred_test = (probs_test >= cutoff).astype(int)
        model_names.append(nazwa)
        sample_names.append(opis)
        accuracy_train.append(accuracy_score(y_true=y_tr, y_pred=y_pred_train))
        accuracy_test.append(accuracy_score(y_true=y_te, y_pred=y_pred_test))
        auc_train.append(roc_auc_score(y_true=y_tr, y_score=probs_train))
        auc_test.append(roc_auc_score(y_true=y_te, y_score=probs_test))
        f1_train.append(f1_score(y_true=y_tr, y_pred=y_pred_train))
        f1_test.append(f1_score(y_true=y_te, y_pred=y_pred_test))
        pot_profit_train.append(potencjalny_zysk_score(y_true=y_tr, y_pred=y_pred_train))
        pot_profit_test.append(potencjalny_zysk_score(y_true=y_te, y_pred=y_pred_test))
        FP = np.sum((y_pred_test == 1) & (y_te == 0))
        TP = np.sum((y_pred_test == 1) & (y_te == 1))
        FN = np.sum((y_pred_test == 0) & (y_te == 1))
        TN = np.sum((y_pred_test == 0) & (y_te == 0))
        false_positive.append(FP)
        false_negative.append(FN)
        true_positive.append(TP)
        true_negative.append(TN)
        recall.append(TP / (TP + FN))
        precision.append(TP / (TP + FP))


podsumowanie_DEV=pd.DataFrame({
                           "Acc_train":accuracy_train, "Acc_test":accuracy_test,
                           "AUC_train":auc_train, "AUC_test":auc_test,
                           "F1_train":f1_train, "F1_test":f1_test, 
                           "Pot_profit_train":pot_profit_train, "Pot_profit_test":pot_profit_test,
                           "false_positive": false_positive,"false_negative": false_negative,
                           "true_positive": true_positive, "true_negative": true_negative,
                           "recall":recall, "precision":precision})

Czysta próbka
*****************************************************
Obecnie wykonuje się model: Regresja logistyczna
*****************************************************
*****************************************************
Obecnie wykonuje się model: Lasy losowe
*****************************************************
*****************************************************
Obecnie wykonuje się model: Regresja logistyczna (bez regularyzacji)
*****************************************************


C:\Users\Marcel\AppData\Roaming\Python\Python313\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


In [3]:
podsumowanie_DEV

,Acc_train,Acc_test,AUC_train,AUC_test,F1_train,F1_test,Pot_profit_train,Pot_profit_test,false_positive,false_negative,true_positive,true_negative,recall,precision
0,0.687513,0.692620,0.748977,0.747773,0.260263,0.260516,0.034359,0.019637,25904,2453,4995,58902,0.670650,0.161656
1,0.704878,0.691287,0.811666,0.744847,0.297959,0.259375,0.239248,0.016080,26019,2461,4987,58787,0.669576,0.160840
2,0.687258,0.692339,0.748977,0.747772,0.260204,0.260455,0.034758,0.020120,25933,2450,4998,58873,0.671053,0.161585


In [4]:
import statsmodels.api as sm

# Przetransformowane dane z pipeline
best_logit = best_estimators[0]
X_train_transformed = best_logit.named_steps["preprocessor"].transform(X_train)
feature_names = best_logit.named_steps["preprocessor"].get_feature_names_out()

# Dopasuj statsmodels Logit
X_const = sm.add_constant(X_train_transformed)
logit_sm = sm.Logit(y_train, X_const).fit(disp=0, maxiter=1000)

# Tabela: współczynniki + p-value
df_coefs = pd.DataFrame({
    "Zmienna": ["const"] + list(feature_names),
    "Współczynnik": logit_sm.params,
    "Błąd std.": logit_sm.bse,
    "z": logit_sm.tvalues,
    "p-value": logit_sm.pvalues
}).sort_values("p-value")

df_coefs["Istotność"] = df_coefs["p-value"].apply(
    lambda p: "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
)

with pd.option_context("display.max_rows", None):
    display(df_coefs)


,Zmienna,Współczynnik,Błąd std.,z,p-value,Istotność
x30,num__EXT_SOURCE_3_imp_education,-0.490308,0.008580,-5.714816e+01,0.000000e+00,***
x21,num__EXT_SOURCE_2,-0.380936,0.008166,-4.664962e+01,0.000000e+00,***
x28,num__EXT_SOURCE_1_imp_income_type_gender,-0.298859,0.013002,-2.298487e+01,6.605559e-117,***
x33,num__LOAN_TO_GOOD,0.171404,0.008655,1.980363e+01,2.770055e-87,***
x27,num__EXT_SOURCE_1_missing,0.125647,0.009270,1.355402e+01,7.501170e-42,***
x31,num__DAYS_EMPLOYED_imp_income,0.151318,0.011232,1.347206e+01,2.284143e-41,***
x2,num__FLAG_OWN_CAR,-0.117933,0.009276,-1.271362e+01,4.967877e-37,***
x24,num__edu_higher_academic,-0.120604,0.010173,-1.185553e+01,2.014524e-32,***
x26,num__DEF_60_CNT_SOCIAL_CIRCLE_eq_0,-0.072861,0.007985,-9.125177e+00,7.161917e-20,***
x12,num__FLAG_WORK_PHONE,0.073489,0.008783,8.367053e+00,5.907737e-17,***


In [ ]:
import joblib

# Zapisz główny model (FIX: best_estimators zamiast model)
joblib.dump(best_estimators[0], 'best_credit_model.pkl')

# Zapisz wszystkie modele do porównania
all_models = {
    "Regresja logistyczna": best_estimators[0],
    "Lasy losowe": best_estimators[1],
    "Reg. log. (bez reg.)": best_estimators[2],
}
joblib.dump(all_models, 'all_models.pkl')


['best_credit_model.pkl']